#### Relevant imports

# Notebook 7 — Hybrid Search: Categorical Filtering + Semantic Similarity

## What You Will Learn

Keyword and category filters alone miss relevant results when users phrase things differently. Semantic search alone ignores hard constraints like product type. **Hybrid search** combines both — first filter categorically, then rank by meaning. This notebook builds a product recommendation pipeline using this approach.

### Pipeline

```
User Query
    │
    ▼
Semantic Match → Identify Product Category (threshold filter)
    │
    ▼
SQL Query → Fetch all products in that category
    │
    ▼
Semantic Rank → Re-order by similarity to user query (top-k)
    │
    ▼
LLM → Final recommendation answer
```

### Topics Covered

- **Sentence Transformers** — local embedding models (`all-MiniLM-L6-v2`) for fast semantic search
- **Cosine Similarity** — computing and thresholding similarity scores with `sentence_transformers.util`
- **Threshold Filtering** — only keeping results above a minimum similarity score
- **Top-K Selection** — picking the most relevant rows for LLM context
- **CSV Utility Functions** — `result_to_csv_string`, `extract_column_from_csv`, `filter_csv_rows_by_index`

### Skills You Will Build

- Load and use a local sentence transformer model
- Build a reusable `semantic_similarity_rank()` function with threshold support
- Chain categorical SQL filtering with semantic re-ranking
- Use CSV as a compact context format to avoid token limit issues
- Build an end-to-end product recommendation RAG system

> **Why it matters:** Real-world queries are ambiguous. Hybrid search is the standard pattern for production RAG systems dealing with product catalogues, documents, and structured records.

In [1]:
import csv
import io
from dotenv import load_dotenv
from hf_llm import hf_chat_completion
from sqlalchemy import create_engine, text, Result
import os

/Users/krypticmouse/Downloads/Week6/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from sentence_transformers import SentenceTransformer, util

>Create a Groq Client

In [3]:
load_dotenv()
HF_Model = "google/flan-t5-base"

**Embedder**  
Embedder model is used from sentence transformer  
This is used for making semantic search

In [4]:
# Load embedder model
# model = SentenceTransformer('all-mpnet-base-v2')
model = SentenceTransformer('all-MiniLM-L6-v2')

**Similarity Search**  
This function makes the semantic similarity search of a string  
It searches the strings that are closely related to search string by meanining  
The reference strings re-ordered based on the similarity  
If there is a distance threshold provided, only the ones relevant are provided

In [ ]:
def semantic_similarity_rank(
    search_string: str,  # search query
    sentences: list[str], # list of sentences to search from
    threshold: float = 0.0 # filtering threshold
) -> tuple[list[str], list[int]]:
    """
    Ranks sentences based on semantic similarity to the search_string.

    Args:
        search_string (str): The input query.
        sentences (list[str]): List of sentences to compare.
        threshold (float): Similarity threshold (0 means no threshold).

    Returns:
        tuple: (reordered_sentences, original_indexes)
    """

    # Encode search string and sentence list
    search_embedding = model.encode(search_string, convert_to_tensor=True)
    sentence_embeddings = model.encode(sentences, convert_to_tensor=True)

    # Compute cosine similarity score
    cosine_scores = util.cos_sim(search_embedding, sentence_embeddings)[0]

    # Pair sentences with scores and original indices
    indexed_scores = [
        (i, s, float(score)) for i, (s, score) in enumerate(zip(sentences, cosine_scores))
        if threshold == 0.0 or float(score) >= threshold
    ]

    # Sort by score descending
    indexed_scores.sort(key=lambda x: x[2], reverse=True)

    # Extract reordered sentences and original indices
    reordered_sentences = [s for _, s, _ in indexed_scores]
    original_indexes = [i for i, _, _ in indexed_scores]

    # Output the reordered senteces and the re-ordered indices in original set
    return reordered_sentences, original_indexes
    
    # # Additional output
    # scores = [sc[2] for sc in indexed_scores]
    # return reordered_sentences, original_indexes, scores


>Let's try using it for sentence similarity search.  
>Some sentences are provided with varied meaning  
>See how it picks the relevance with search string

In [6]:
# Example usage of semantic similarity
sentences = ["Software update improved cloud data processing.",
                "Researchers study atmospheric carbon capture.",
                "Legislators debated new trade agreement's economic impact.",
                "Personalized medicine targets cancer with genetics.",
                "Oil price swings impact consumer spending.",
                "Community programs address urban food deserts.",
                "Adaptive learning boosts STEM student engagement.",
                "Basel exhibition challenged art perceptions.",
                "Heavy monsoon rains disrupted regional transport.",
                "Court ruling set AI content IP precedent.",
                "Optimizing supply chain cuts production costs.",
                "Martian rover data hints at subsurface ice.",
                "Phishing attack compromised corporate credentials.",
                "Smart cities prioritize green transport.",
                "Drought-resistant crops ensure food security.",
                "Underdog team's win made sports headlines.",
                "Pompeii finds reveal Roman daily life.",
                "Sustainable tourism gains eco-conscious travelers.",
                "Cognitive biases influence financial decisions.",
                "Report found discrepancies in project spending."]

search = 'AI is interesting'
# search = 'It might rain tomorrow'

# Make a meaning search
sentence_re_ordered, index_re_oredered = semantic_similarity_rank (search, sentences)

# Check the order of relevance
# print (scores)
print (index_re_oredered)
for snt in sentence_re_ordered :

    print (snt)


[9, 16, 7, 0, 6, 18, 11, 13, 15, 2, 1, 12, 17, 5, 4, 3, 19, 8, 10, 14]
Court ruling set AI content IP precedent.
Pompeii finds reveal Roman daily life.
Basel exhibition challenged art perceptions.
Software update improved cloud data processing.
Adaptive learning boosts STEM student engagement.
Cognitive biases influence financial decisions.
Martian rover data hints at subsurface ice.
Smart cities prioritize green transport.
Underdog team's win made sports headlines.
Legislators debated new trade agreement's economic impact.
Researchers study atmospheric carbon capture.
Phishing attack compromised corporate credentials.
Sustainable tourism gains eco-conscious travelers.
Community programs address urban food deserts.
Oil price swings impact consumer spending.
Personalized medicine targets cancer with genetics.
Report found discrepancies in project spending.
Heavy monsoon rains disrupted regional transport.
Optimizing supply chain cuts production costs.
Drought-resistant crops ensure food

In [7]:
sentences = ['This phone has good battery', 
             'Laptop gets slow after a while',
             'My PC is not very efficient',
             'My smart phone has clarity of sound']

search = 'Too much screen time is not good'

# Make a meaning search
sentence_re_ordered, index_re_oredered = semantic_similarity_rank (search, sentences)

# Check the order of relevance
# print (scores)
print (index_re_oredered)
for snt in sentence_re_ordered :

    print (snt)

[2, 1, 0, 3]
My PC is not very efficient
Laptop gets slow after a while
This phone has good battery
My smart phone has clarity of sound


**Utility**  
Util functions that are required for hadling CSV results  
CSV format is chosen considering the number of rows that needs to be handled by LLM  
if JSON, the context may be oversized and limits might hit

In [8]:
def result_to_csv_string(result: Result, delimiter: str = ',') -> tuple[str, int]:
    """
    Convert qeury result to CSV-formatted string.

    Args:
        result (Result): The result of conn.execute().
        delimiter (str): Delimiter used in CSV (default is comma).

    Returns:
        tuple: (CSV string including headers, number of data rows)
    """

    # Get column names
    headers = result.keys()

    # Get all rows
    rows = result.fetchall()

    # Use StringIO to build CSV string
    output = io.StringIO()
    writer = csv.writer(output, delimiter=delimiter)

    # Write headers and data rows
    writer.writerow(headers)
    writer.writerows(rows)

    csv_string = output.getvalue()
    row_count = len(rows)

    return csv_string, row_count


def extract_column_from_csv(csv_text: str, column_name: str, delimiter: str = ',') -> list[str]:
    """
    Extracts a column from CSV text as a list of strings based on the column header.

    Args:
        csv_text (str): The full CSV content as string.
        column_name (str): The name of the column to extract.
        delimiter (str): The CSV delimiter (default is ',').

    Returns:
        list[str]: List of values from the specified column.
    """
    reader = csv.DictReader(io.StringIO(csv_text), delimiter=delimiter)
    return [row[column_name] for row in reader if column_name in row]


def filter_csv_rows_by_index(csv_text: str, row_indexes: list[int], delimiter: str = ',') -> str:
    """
    Filters specific data rows from CSV text by their row index (excluding the header row).

    Args:
        csv_text (str): The full CSV content as string.
        row_indexes (list[int]): List of 0-based row numbers (excluding header).
        delimiter (str): The CSV delimiter (default is ',').

    Returns:
        str: New CSV string with only selected rows (including header).
    """
    reader = csv.reader(io.StringIO(csv_text), delimiter=delimiter)
    rows = list(reader)
    
    if not rows:
        return ""

    header = rows[0]
    data_rows = rows[1:]

    selected_rows = [data_rows[i] for i in row_indexes if 0 <= i < len(data_rows)]

    output = io.StringIO()
    writer = csv.writer(output, delimiter=delimiter)
    writer.writerow(header)
    writer.writerows(selected_rows)

    return output.getvalue()

**Instructions**
Query instruction is adapted to consider the product choice of user along with question / prompt  

In [9]:
R_Instr = "Using the context given, provide response to the user question or statement.\
            Context is provided as CSV formatted string.\
            Answer to the question with details"

>SQLite Engine with connection created  
>The product catalogue database is used

In [12]:
# There is an engine instance created, which can handle multiple connetions
DB_File = "Sample_3.db"

if os.path.exists (DB_File):
    sql_engine = create_engine("sqlite:///"+DB_File)
    conn = sql_engine.connect ()
else:
    print ("DB Files does not exist")

**Product category**  
From the user query, identify product category by making semantic search  
The distinct values of the product type is matched to the user prompt  
Depending on if there is a relevant product type (threshold is given) the product selection is made

In [ ]:
# Prompt = "I need a ear phone which has durable battery"
Prompt = "Give me some latest monitor that has good brightness"
# Prompt = "Can you suggest me a Phone that has sensitive touch screen and durable battery?"
# Prompt = "Which is the good TV?"

# Identify Product type by semantic search

# get all the product categories and then semantically search thru them corresp to the search query 
result = conn.execute(text(f"""
                            SELECT DISTINCT Product_Type from product_catalogue
                              """))

# Get the Product categories from database and make it into a list
rows = result.fetchall()
Categories = [str(row[0]) for row in rows]
# print (Categories)

Shortlist, Order = semantic_similarity_rank (Prompt, Categories, 0.3)
Shortlist


['Monitor', 'Laptop']

**Query**  
Filter based on the product type that is identified.  
The result is then fed for further semantic search

In [14]:
# Query from DB for the Product type that is identified
result = conn.execute (text(f"""
                            SELECT * from product_catalogue WHERE Product_Type = '{Shortlist[0]}'
                              """))

# The output is then convereted into CSV text
CSV_Result, Nb_Rows = result_to_csv_string (result)
print (Nb_Rows)

# Extract the User Feedback Column
Feedback = extract_column_from_csv (CSV_Result, 'User_Feedback')



93


**Semantic Search**  
Since the feedback column is textual data, semantic searach is applied to identify the relevant ones  
Top k numbers are then filtered based on the re-ordered ranking  
This is used as context to LLM for answering user query  

In [15]:
# How many to be filtered
top_k = 10

# Make semantic similarity in feedback column and get the order by relevance
Shortlist, Order = semantic_similarity_rank (Prompt, Feedback)
print (Order)

# Filter out the required numbers. this is the indices that is required after the re-ordering
Filter = Order[:top_k]

# Filter the CSV content by required row numbers
Context = filter_csv_rows_by_index (CSV_Result, Filter)
# print (Context)


[18, 12, 87, 65, 13, 4, 39, 74, 76, 67, 15, 22, 90, 30, 69, 88, 32, 72, 86, 57, 63, 14, 26, 37, 48, 83, 35, 53, 54, 70, 73, 91, 2, 38, 27, 79, 23, 58, 51, 8, 45, 21, 68, 81, 52, 89, 34, 84, 64, 6, 25, 46, 55, 20, 49, 11, 77, 66, 40, 62, 9, 29, 50, 56, 92, 75, 78, 33, 80, 85, 16, 41, 1, 42, 82, 71, 17, 24, 7, 19, 5, 10, 59, 44, 61, 60, 43, 0, 36, 31, 47, 3, 28]


**LLM answer**
With the context that is categorically and semantically filtered, it is then provided to LLM  
Along with this context user query is responded 

In [16]:
messages=[
    {
        "role": "system",
        "content": R_Instr
    },

    {
        "role": "user",
        "content":"Context : \n"+ Context + "Query : \n" + Prompt
    }
]
completion = hf_chat_completion(
    messages=messages,
    model="openai/gpt-oss-120b",
)

print (completion.choices[0].message.content)

Here are the newest monitors in the list that are praised for **exceptional brightness** (the user‑feedback column explicitly calls out “Exceptional brightness and vivid display” for every product).  
I’ve selected the ones with the most recent launch years – 2025 and the next‑most‑recent 2024 – and included their key specs and price so you can compare them easily.

| Year | Brand | Model | Size | Resolution | Refresh Rate | Price (USD) | Why it’s bright |
|------|-------|-------|------|------------|--------------|-------------|-----------------|
| **2025** | Clarovue | **CLA5884** | 27 in | 1440p | 77 Hz | $713.96 | “Exceptional brightness and vivid display make visuals pop.” |
| **2025** | Viewlet | **VIE3927** | 28 in | 1440p | 145 Hz | $531.60 | “Exceptional brightness and vivid display make visuals pop.” |
| **2025** | Viewlet | **VIE3626** | 23 in | 1440p | 116 Hz | $781.13 | “Exceptional brightness and vivid display make visuals pop.” |
| **2024** | Displion | **DIS1661** | 25 i